# Survival Simulator - baseline runner

Execution trigger only: all logic lives in `agents/` and `training/` (.py files).

* **Scored runs** (logged to `results/index.csv`) are launched as a **fresh subprocess** via `!python -m training.evaluate`, so kernel state can never leak into a result.
* **Rendering** runs in-kernel and is for watching the game only; its scores are never logged.

Run this notebook with its working directory = `survival-simulator/`.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
CLONE_DIR = "/home/jovyan/Nordic-AI-cup-2026"
if os.path.isdir(os.path.join(CLONE_DIR, ".git")):
    print(f"{CLONE_DIR} already cloned - skipping (use `git pull` there to update)")
else:
    # GitHub token is read from a git-ignored .env (GITHUB_TOKEN=...) in the kernel's cwd, or from the environment
    if os.path.isfile(".env"):
        for line in open(".env"):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#"):
                os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
    TOKEN = os.environ.get("GITHUB_TOKEN")
    if not TOKEN:
        raise RuntimeError(f"GITHUB_TOKEN not set - create {os.path.abspath('.env')} containing GITHUB_TOKEN=<token>")
    !git clone https://{TOKEN}@github.com/sjoeen/Nordic-AI-cup-2026.git {CLONE_DIR}


/home/jovyan/Nordic-AI-cup-2026 already cloned - skipping (use `git pull` there to update)


In [3]:
import os
os.chdir("/home/jovyan/Nordic-AI-cup-2026")
!git checkout challenge-1
print(os.listdir("."))

M	Nordic-AI-Cup-2026-main/survival-simulator/results/index.csv
Already on 'challenge-1'
Your branch is up to date with 'origin/challenge-1'.
['trial_racecar', '.git', '.gitignore', 'test.ipynb', 'Nordic-AI-Cup-2026-main', 'main.py', '.gitattributes']


In [4]:
import os, sys, subprocess, glob


# Must run from survival-simulator/ so `src`, `agents`, `training` import.
# JupyterHub kernels may start in $HOME, so locate the folder and cd into it.
if os.path.basename(os.getcwd()) != "survival-simulator":
    candidates = sorted({os.path.realpath(p) for p in glob.glob(os.path.join(os.getcwd(), "**", "survival-simulator"), recursive=True)
                         if os.path.isfile(os.path.join(p, "requirements.txt"))})
    if len(candidates) != 1:
        raise RuntimeError(f"cwd is {os.getcwd()}; found {len(candidates)} survival-simulator checkouts {candidates} - %cd into the right one")
    os.chdir(candidates[0])
print("cwd:", os.getcwd())
sys.path.insert(0, os.getcwd())
print(sys.executable)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
# Warn if this kernel's packages differ from the pinned simulator requirements
from importlib.metadata import version, PackageNotFoundError
for line in open("requirements.txt"):
    name, _, pinned = line.strip().partition("==")
    try:
        installed = version(name)
    except PackageNotFoundError:
        installed = "MISSING"
    if installed != pinned:
        print(f"WARNING: {name} installed={installed} pinned={pinned}")


cwd: /home/jovyan/Nordic-AI-cup-2026/Nordic-AI-Cup-2026-main/survival-simulator
/opt/conda/bin/python
4302f8d [NONE] docs: handover, correct origin of duplicate result rows



## 0. One-time install (skip if already installed)

In [5]:
#!{sys.executable} -m pip install -r requirements.txt -r requirements-dev.txt

### 0b. Fix: RecursionError in `numpy.__getattr__("rec")` (hit by pandas `read_csv`/`isna`)

Seen after the cluster image was upgraded: `numpy.rec` fails to import cleanly, so numpy's
lazy `__getattr__` keeps recursing trying to resolve it, until pandas' `isna()` blows the
stack. `pip show`/version-pin checks won't catch this (metadata says 2.3.5, matching
`requirements.txt`, but the install itself is corrupted/partial) -- only a forced clean
reinstall fixes it. Run the next cell once if you hit this.

In [6]:
import numpy, pandas
print("before: numpy", numpy.__version__, numpy.__file__)
print("before: pandas", pandas.__version__, pandas.__file__)

!{sys.executable} -m pip install --force-reinstall --no-cache-dir "numpy==2.3.5" pandas

print("Restart this kernel now (numpy/pandas were reloaded in-process; a live re-import won't")
print("pick up the reinstalled files cleanly) then re-run from the top.")

before: numpy 2.3.5 /opt/conda/lib/python3.12/site-packages/numpy/__init__.py
before: pandas 3.0.6 /opt/conda/lib/python3.12/site-packages/pandas/__init__.py
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 410.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 301.0 MB/s  0:00:00
  Attempting uninstall: six
    Found existing installation: six 1.17.0
    Uninstalling six-1.17.0:
      Successfully uninstalled six-1.17.0━━━━━━━ 0/4 [six]
  Attempting uninstall: numpy━━━━━━━━━━━━━━━ 0/4 [six]
    Found existing installation: numpy 2.3.5 0/4 [six]
    Uninstalling numpy-2.3.5:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [numpy]
      Successfully uninstalled numpy-2.3.5━━━━━━━━━━━━━━━━━━━━ 1/4 [numpy]
  Attempting uninstall: python-dateutil━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [numpy]
    Found existing installation: python-dateutil 2.9.0.post0━━ 1/4 [numpy]
    Uninstalling python-dateutil-2.9.0.post0:━━━━━━━━━━━━━━━━━ 1/4 [numpy]
      Successfully uninstalled python-dateu

## 1. Tests (run before every scored run)

In [7]:
!{sys.executable} -m pytest -q tests

[autoreload of six failed: Traceback (most recent call last):
  File "/opt/conda/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 325, in check
    superreload(m, reload, self.old_objects)
  File "/opt/conda/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 621, in superreload
    update_generic(old_obj, new_obj)
  File "/opt/conda/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 447, in update_generic
    update(a, b)
  File "/opt/conda/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 380, in update_class
    old_obj = getattr(old, key)
              ^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/six.py", line 98, in __get__
    setattr(obj, self.name, result)  # Invokes __set__.
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'NoneType' object has no attribute 'cStringIO'. Did you mean: 'StringIO'?
]
[autoreload of dateutil.tz._factories failed: Traceback (most recent call last):
  F

.....                                                                    [100%]
=============================== warnings summary ===============================
tests/test_agents.py::test_episode_seed_is_deterministic
  /opt/conda/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
    from pkg_resources import resource_stream, resource_exists

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
5 passed, 1 warning in 3.86s


## 2. Scored evaluation (fresh process)

Parameters to set per experiment: experiment id, config, seeds, workers (cluster: 4 CPUs), per-game wall-clock cap.
Commit before running - the run records the git revision and flags `-dirty` trees.

In [8]:
EXPERIMENT = "C1-E00"
CONFIG = "training/configs/dummy_v0.json"
SEEDS = "0 1 2 3 4"
WORKERS = 4
MAX_WALL_SEC = 1800

!{sys.executable} -m training.evaluate --experiment {EXPERIMENT} --config {CONFIG} --seeds {SEEDS} --workers {WORKERS} --max-wall-sec {MAX_WALL_SEC}

run_id=C1-E00_dummy_v0_20260918-083606 config=training/configs/dummy_v0.json hash=e27e72fb908f rev=4302f8d-dirty seeds=[0, 1, 2, 3, 4] workers=4
python=3.12.11 exe=/opt/conda/bin/python numpy=2.3.5 scipy=1.16.3 shapely=2.1.2 pygame=2.6.1 pydantic=2.12.4
/opt/conda/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
/opt/conda/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
/opt/conda/lib/p

In [9]:
EXPERIMENT = "C1-E01"
CONFIG = "training/configs/heuristic_v0.json"

!{sys.executable} -m training.evaluate --experiment {EXPERIMENT} --config {CONFIG} --seeds {SEEDS} --workers {WORKERS} --max-wall-sec {MAX_WALL_SEC}

run_id=C1-E01_heuristic_v0_20260918-083611 config=training/configs/heuristic_v0.json hash=2793444c26b8 rev=4302f8d-dirty seeds=[0, 1, 2, 3, 4] workers=4
python=3.12.11 exe=/opt/conda/bin/python numpy=2.3.5 scipy=1.16.3 shapely=2.1.2 pygame=2.6.1 pydantic=2.12.4
/opt/conda/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
/opt/conda/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
/opt/con

## 3. Results index

In [ ]:
import pandas as pd
pd.read_csv("results/index.csv")

## 4. Watch a game (inspection only, not scored)

`max_sim_time` limits how much of the game is rendered (one frame per simulated second by default).

In [ ]:
from IPython.display import Video, Image, display
from training.render import render_episode

video_path, stats = render_episode("training/configs/heuristic_v0.json", seed=0, max_sim_time=300, frame_every=10, width=640, fps=20)
display(Video(os.path.relpath(video_path), embed=False) if video_path.endswith(".mp4") else Image(filename=video_path))
stats

## 5. External finalists comparison (Dispersal vs heuristic_v1, seeds 20-39)

Runs Astra's `agent_dispersal` finalist candidate against our `heuristic_v1` on 20 unseen
seeds (not the 1/2/3 they were tuned/compared on), through the same headless game loop as
`training/batch_runner.py`. The finalist's own `decide_all()` is called unmodified via a
thin adapter in `external/finalists/run_external.py`.

**Requires `challenge-1V2` to be pushed to origin first** (it pulls `external/finalists/`,
`agents/heuristic_v1.py`, and `training/configs/heuristic_v1.json` from there via a
path-scoped checkout, without switching this kernel off whatever branch it's currently on).

In [ ]:
!git fetch origin challenge-1V2
!git checkout origin/challenge-1V2 -- external/finalists agents/heuristic_v1.py training/configs/heuristic_v1.json

SEEDS = " ".join(str(s) for s in range(20, 40))
!{sys.executable} external/finalists/run_external.py --agents dispersal v1 --seeds {SEEDS} --out results/EXTERNAL_seeds20-39.csv

In [ ]:
import pandas as pd
df = pd.read_csv("results/EXTERNAL_seeds20-39.csv")
display(df.groupby("agent")["score"].agg(["mean", "median", "std", "count"]))
df

## 6. External candidate comparison (original-eat-rest-overcrowding-v4 vs heuristic_v1, seeds 1-3)

Runs the `original-eat-rest-overcrowding-v4` candidate (`survival_agent.py`'s
`OvercrowdingLurePolicy` via `make_policy()`) against our `heuristic_v1` on seeds 1-3,
through the same headless game loop as `training/batch_runner.py`. The candidate's own
`decide_all(states, sim_time)` is called unmodified via a thin adapter in
`external/candidates/run_candidate.py`.

The candidate package's own `validation.json` reports `full_simulations_run: 0` -- its 9
checks compare decision state against the baseline, they are not survival scores. This
cell is the first run of this code through an actual full game.

**Requires `challenge-1V2` to be pushed to origin first** (it pulls
`external/candidates/`, `agents/heuristic_v1.py`, and `training/configs/heuristic_v1.json`
from there via a path-scoped checkout, without switching this kernel off whatever branch
it's currently on).

In [ ]:
!git fetch origin challenge-1V2
!git checkout origin/challenge-1V2 -- external/candidates external/finalists agents/heuristic_v1.py training/configs/heuristic_v1.json

SEEDS = "1 2 3"
!{sys.executable} external/candidates/run_candidate.py --seeds {SEEDS} --out results/EXTERNAL_original-eat-rest-overcrowding-v4_seeds1-3.csv
!{sys.executable} external/finalists/run_external.py --agents v1 --seeds {SEEDS} --out results/EXTERNAL_v1_seeds1-3.csv

In [ ]:
import pandas as pd
df_v4 = pd.read_csv("results/EXTERNAL_original-eat-rest-overcrowding-v4_seeds1-3.csv")
df_v1 = pd.read_csv("results/EXTERNAL_v1_seeds1-3.csv")
df = pd.concat([df_v4, df_v1], ignore_index=True)
display(df.groupby("agent")["score"].agg(["mean", "median", "std", "count"]))
df